### Imports

In [1]:
import os
import pandas as pd
import json
from tokenizers import Tokenizer # main object for tokenization
from tokenizers.models import BPE # model implementing Byte Pair Encoding.
from tokenizers.trainers import BpeTrainer # trains the BPE vocabulary
from tokenizers.pre_tokenizers import Whitespace # simple pre-tokenizer splitting on spaces

### Settings

In [2]:
DIR_DATA = "data"
DIR_TEXTS = os.path.join(DIR_DATA, "texts")
FILE_METADATA = os.path.join(DIR_DATA, "metadata.csv")

### Tokenization

In [3]:
tokenizer = Tokenizer(BPE())           # BPE model with unknown token
tokenizer.pre_tokenizer = Whitespace() # Split text by spaces first

In [4]:
# Load the cleaned corpus
with open(os.path.join(DIR_DATA, "cleaned_corpus.txt"), "r", encoding="utf-8") as f:
    corpus = f.read()

In [5]:
# Training the tokenizer on the corpus.
VOCAB_SIZE = 25000          # Educated guess based on the number of unique tokens and the long tail distribution of token counts
MIN_FREQUENCY = 1          # Minimum frequency for a token to be included in the vocabulary, based on the distribution of token counts and the large number of singular tokens in the corpus
SPECIAL_TOKENS = []         # No special tokens for now, but we could add [UNK], [PAD], etc. if needed
END_OF_WORD_SUFFIX = "</w>" # Suffix to indicate the end of a word, commonly used in BPE tokenization to distinguish between subword tokens and whole word tokens
trainer = BpeTrainer(vocab_size=VOCAB_SIZE, min_frequency=MIN_FREQUENCY, special_tokens=SPECIAL_TOKENS, end_of_word_suffix=END_OF_WORD_SUFFIX)

# FIX: Wrap corpus in a list so it is treated as a single document/sequence, not a list of characters
tokenizer.train_from_iterator([corpus], trainer=trainer)

vocab_df = pd.DataFrame(tokenizer.get_vocab().items(), columns=["token", "id"]).sort_values("id").reset_index(drop=True)
print(f"Sample of the learned BPE vocabulary:")
print(vocab_df.sample(10))




Sample of the learned BPE vocabulary:
                token     id
8816        waked</w>   8816
8524   hesitation</w>   8524
8218           identi   8218
3122          ace</w>   3122
16287         obsequi  16287
10933   furiously</w>  10933
439           old</w>    439
9418     marching</w>   9418
4691       brutus</w>   4691
2973         cles</w>   2973


In [6]:
# Example of encoding some words using the trained tokenizer
print(f"Encoding some sample words using the trained BPE tokenizer:")
for words in ["bathroom", "scientology", "lowest", "absolutely", "nocap", "threehundredandthirtythreetrillionth"]:
    print(f"    {words:<36} => {'-'.join(tokenizer.encode(words).tokens)}")

Encoding some sample words using the trained BPE tokenizer:
    bathroom                             => ba-th-room</w>
    scientology                          => sci-en-to-logy</w>
    lowest                               => lowest</w>
    absolutely                           => absolutely</w>
    nocap                                => no-cap</w>
    threehundredandthirtythreetrillionth => thre-e-hundre-dan-d-thir-ty-thre-e-tri-lli-onth</w>


In [7]:
# Save then trained tokenizer
tokenizer.save("bpe_tokenizer.json")

In [8]:
# Encode the corpus using the trained tokenizer (could take a couple minutes)
# " ".join(corpus) was inserting spaces between every character if corpus is a string!
# If corpus is already a string, just pass it directly.
encoded_corpus = tokenizer.encode(corpus)

In [9]:
# Try detokenizing a sample of the encoded corpus to verify it works correctly
sample_encoded = encoded_corpus.ids[:100] # Take the first 100 token IDs as a sample
detokenized_sample = tokenizer.decode(sample_encoded)  
print(f"\nSample of the encoded corpus (first 100 token IDs):\n{sample_encoded}")
print(f"\nDetokenized sample:\n{detokenized_sample}")


Sample of the encoded corpus (first 100 token IDs):
[86, 3207, 264, 210, 124, 288, 174, 1641, 3411, 1887, 5459, 1283, 1283, 1283, 1283, 1283, 656, 83, 123, 83, 2078, 83, 83, 150, 256, 7743, 94, 911, 118, 243, 11621, 368, 3453, 86, 12465, 93, 125, 9357, 183, 150, 179, 3263, 134, 356, 2072, 15844, 4645, 83, 66, 2031, 514, 89, 156, 440, 4010, 119, 94, 3148, 156, 814, 1726, 93, 156, 9856, 89, 7551, 3396, 100, 86, 2826, 93, 156, 9899, 83, 66, 339, 990, 789, 1290, 93, 89, 104, 66, 1975, 100, 86, 3783, 93, 66, 1136, 57, 1725, 7517, 8777, 1892, 374, 156, 183, 23971, 156]

Detokenized sample:
the</w> modern</w> pro me the us</w> by</w> mary</w> shel ley</w> contents</w> letter</w> letter</w> letter</w> letter</w> letter</w> mrs</w> .</w> st</w> .</w> dec</w> .</w> .</w> you</w> will</w> rejoice</w> to</w> hear</w> that</w> no</w> disaster</w> has</w> accompanied</w> the</w> commencement</w> of</w> an</w> enterprise</w> which</w> you</w> have</w> regarded</w> with</w> such</w> evil</w> forebo d

In [10]:
# Save the encoded corpus
with open("encoded_text.json", "w") as f:
    json.dump(encoded_corpus.ids, f)